# GPU で楽曲をステム分離

[Demucs の推論用パッケージ](https://github.com/openmirlab/demucs-infer) の `htdemucs` モデルを使い、**drums / bass / other / vocals** の4本に分離します。MuScripterの入力候補となる「ドラム以外」の確認にも使えますが、分離音声の品質は音源によって変わります。

Colab の **ランタイム → ランタイムのタイプを変更 → GPU** を選び、順に実行してください。モデルの重みは初回にダウンロードされます。

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU がありません。「ランタイム → ランタイムのタイプを変更 → GPU」を選択してください。')
print('GPU:', torch.cuda.get_device_name(0))
print('空きVRAM: %.1f GiB' % (torch.cuda.mem_get_info()[0] / 1024**3))


## ライブラリを準備

In [ ]:
%pip -q install "demucs-infer>=4.2,<5"


## 音声を1つアップロード

長い曲は処理時間と出力ZIPの容量が大きくなります。まず短い音源で試せます。

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
import tempfile

if not shutil.which('ffmpeg'):
    raise RuntimeError('ffmpeg がありません。')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('音声を1つだけ選んでください。')
name, data = next(iter(uploaded.items()))
suffix = Path(name).suffix.lower()
if suffix not in {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}:
    raise ValueError('WAV / MP3 / FLAC / OGG / M4A を選んでください。')
if len(data) > 100 * 1024**2:
    raise ValueError('100 MiB 以下の音声を選んでください。')
work_dir = Path(tempfile.mkdtemp(prefix='demucs_stems_', dir='/content'))
audio_path = work_dir / ('input' + suffix)
audio_path.write_bytes(data)
print('入力:', Path(name).name)


## 4ステムを生成し、ZIPで取得

出力はWAVです。Colab の空き容量とブラウザーのダウンロード容量に注意してください。

In [ ]:
from demucs_infer import DemucsSession
from demucs_infer.audio import save_audio
import zipfile

stem_dir = work_dir / 'stems'
stem_dir.mkdir()
with DemucsSession(model='htdemucs', device='cuda') as session:
    mixture, stems = session.infer(audio_path)
    sample_rate = session.samplerate
    for stem_name, waveform in stems.items():
        path = stem_dir / f'{stem_name}.wav'
        save_audio(waveform.cpu(), str(path), sample_rate)
        print(stem_name, round(path.stat().st_size / 1024**2, 1), 'MiB')

zip_path = work_dir / 'stems.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(stem_dir.glob('*.wav')):
        archive.write(path, arcname=path.name)
files.download(str(zip_path))
